# T3 SigLIP — Flickr30k decoder adaptation pilot seed 42

Pilot này là gate bắt buộc trước full run. Ba pilot T1, T2 và T3 độc lập và có
thể chạy song song trong ba phiên Kaggle. Không đọc Validation caption và không
mở Held-out Test.

## Kaggle input bắt buộc — đúng 6 input

1. Output `F30K_Data`.
2. Output `F30K_Base`.
3. Output `F30K_T3_Handoff`.
4. Output `phase3_teacher_c_seed42`.
5. Dataset `Flickr Image dataset`.
6. Dataset/Kaggle Model SigLIP offline có `MODEL_SOURCE_MANIFEST.json`.

## Runtime

- Accelerator: GPU T4; một GPU là đủ.
- Internet: Off.
- T1/T2: 128 ảnh; T3: 100 optimizer-step.
- Pilot T3 bị hủy trọng số sau đo thời gian; full run phải khởi tạo lại.


## Cell 1 — Input gate, Flickr30k contract và image index

In [1]:

import os
EXPERIMENT_ID = "F30K_T3_Adapt42_Pilot"
TEACHER_ID = "T3"
HANDOFF_RESULT_NAME = "F30K_T3_Handoff_RESULT.json"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"

import csv, hashlib, json, math, os, random, sys, time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch

INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
WORK_ROOT = Path(os.environ.get("KLCN_WORK_ROOT", "/kaggle/working")) / EXPERIMENT_ID
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def sha256(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)

def unique_named(name):
    matches = sorted(path for path in INPUT_ROOT.rglob(name) if path.is_file())
    assert len(matches) == 1, (f"Cần đúng một {name}", [str(x) for x in matches])
    return matches[0]

def image_index(required_names):
    candidates = [p for p in INPUT_ROOT.rglob("*.jpg") if p.name in required_names]
    by_name = {}
    duplicate_aliases = 0
    for path in candidates:
        old = by_name.get(path.name)
        if old is None:
            by_name[path.name] = path
            continue
        if old.resolve() == path.resolve():
            continue
        assert old.stat().st_size == path.stat().st_size, (
            "Ảnh trùng tên nhưng khác nội dung/kích thước", path.name, old, path
        )
        duplicate_aliases += 1
        by_name[path.name] = min(
            (old, path), key=lambda p: (len(p.parts), len(str(p)), str(p))
        )
    missing = sorted(required_names - set(by_name))
    assert not missing, ("Thiếu ảnh Flickr30k", len(missing), missing[:10])
    return by_name, duplicate_aliases

DATA_PATH = unique_named("f30k_train.parquet")
DATA_CONTRACT_PATH = unique_named("F30K_DATA_CONTRACT.json")
BASE_READY_PATH = unique_named("F30K_BASE_READY.json")
VOCAB_PATH = unique_named("f30k_vocabulary_minfreq5.json")
HANDOFF_PATH = unique_named(HANDOFF_RESULT_NAME)

data_contract = read_json(DATA_CONTRACT_PATH)
base_ready = read_json(BASE_READY_PATH)
vocab = read_json(VOCAB_PATH)
handoff = read_json(HANDOFF_PATH)

assert data_contract["status"] == "complete"
assert data_contract["train_image_count"] == 29000
assert data_contract["validation_image_count"] == 1014
assert data_contract["test_image_count"] == 1000
assert data_contract["vocabulary_size"] == 7456
assert data_contract["test_set_status"] == "sealed"
assert data_contract["test_caption_exported"] is False
assert base_ready["status"] == "authorized"
assert base_ready["test_set_status"] == "sealed"
assert base_ready["test_set_used"] is False
assert handoff["teacher_id"] == TEACHER_ID
assert handoff["flickr30k"]["test_set_status"] == "sealed"
assert handoff["flickr30k"]["test_set_used"] is False
assert sha256(DATA_PATH) == data_contract["artifacts"]["train"]["sha256"]
assert sha256(VOCAB_PATH) == data_contract["artifacts"]["vocabulary"]["sha256"]

train_df = pd.read_parquet(DATA_PATH)
assert train_df["image_id"].nunique() == 29000
assert len(train_df) == 145000
train_ids = sorted(int(x) for x in train_df["image_id"].unique())
required_names = {f"{image_id}.jpg" for image_id in train_ids}
IMAGE_BY_NAME, DUPLICATE_ALIASES = image_index(required_names)

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
assert torch.cuda.is_available(), "Notebook pilot yêu cầu GPU T4."
DEVICE = torch.device("cuda:0")

print("Common gate             : PASSED")
print("Experiment              :", EXPERIMENT_ID)
print("Teacher                 :", TEACHER_ID)
print("Flickr30k Train         : 29,000 images")
print("Vocabulary              : 7,456")
print("Nested image aliases    :", DUPLICATE_ALIASES, "(deduplicated)")
print("Held-out Test           : SEALED / NOT READ")


Common gate             : PASSED
Experiment              : F30K_T3_Adapt42_Pilot
Teacher                 : T3
Flickr30k Train         : 29,000 images
Vocabulary              : 7,456
Nested image aliases    : 29000 (deduplicated)
Held-out Test           : SEALED / NOT READ


## Cell 2 — Teacher workload pilot

In [2]:

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from transformers import SiglipModel

assert handoff["status"] == "passed_with_required_adaptation"
assert handoff["method_contract"]["source_vocabulary_size"] == 10080
assert handoff["method_contract"]["target_vocabulary_size"] == 7456
EXPECTED_CHECKPOINT_SHA = handoff["teacher_source"]["development_checkpoint_sha256"]
CHECKPOINT_PATH = unique_named("teacher_c_seed42_best.pt")
assert sha256(CHECKPOINT_PATH) == EXPECTED_CHECKPOINT_SHA

model_sources = []
for manifest_path in INPUT_ROOT.rglob("MODEL_SOURCE_MANIFEST.json"):
    try:
        payload = read_json(manifest_path)
    except Exception:
        continue
    if payload.get("repo_id") == "google/siglip-base-patch16-224":
        model_sources.append(manifest_path.parent)
assert len(model_sources) == 1, model_sources
MODEL_SOURCE = model_sources[0]

transform = transforms.Compose([
    transforms.Resize((224, 224), interpolation=InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])
groups = []
for image_id, group in train_df.groupby("image_id", sort=True):
    groups.append((int(image_id), [list(map(int, x)) for x in group.token_ids]))

class TrainPilot(Dataset):
    def __len__(self): return len(groups)
    def __getitem__(self, index):
        image_id, captions = groups[index]
        tokens = captions[(image_id + index) % len(captions)]
        with Image.open(IMAGE_BY_NAME[f"{image_id}.jpg"]) as image:
            pixels = transform(image.convert("RGB"))
        return pixels, torch.tensor(tokens, dtype=torch.long)

def collate(rows):
    images, captions = zip(*rows)
    maximum = max(len(x) for x in captions)
    tokens = torch.zeros(len(captions), maximum, dtype=torch.long)
    mask = torch.zeros(len(captions), maximum, dtype=torch.bool)
    for index, values in enumerate(captions):
        tokens[index, :len(values)] = values
        mask[index, :len(values)] = True
    return torch.stack(images), tokens, mask

class SiglipVisualAdapter(nn.Module):
    def __init__(self):
        super().__init__()
        self.projection = nn.Linear(768, 256)
        self.position_embedding = nn.Parameter(torch.zeros(1, 196, 256))
        self.output_norm = nn.LayerNorm(256)
    def forward(self, patches):
        assert patches.shape[1:] == (196, 768)
        return self.output_norm(self.projection(patches) + self.position_embedding)

class CaptionDecoder(nn.Module):
    def __init__(self, vocabulary_size):
        super().__init__()
        self.vocabulary_size = vocabulary_size
        self.token_embedding = nn.Embedding(vocabulary_size, 256, padding_idx=0)
        self.position_embedding = nn.Embedding(31, 256)
        self.embedding_dropout = nn.Dropout(0.1)
        layer = nn.TransformerDecoderLayer(
            256, 8, 1024, 0.1, "gelu", batch_first=True, norm_first=True
        )
        self.transformer_decoder = nn.TransformerDecoder(layer, 3)
        self.final_norm = nn.LayerNorm(256)
        self.output_bias = nn.Parameter(torch.zeros(vocabulary_size))
        nn.init.normal_(self.token_embedding.weight, std=256 ** -0.5)
        nn.init.normal_(self.position_embedding.weight, std=0.02)
        with torch.no_grad(): self.token_embedding.weight[0].zero_()
    def forward(self, ids, memory, attention_mask=None):
        n, t = ids.shape
        positions = torch.arange(t, device=ids.device).unsqueeze(0).expand(n, -1)
        target = self.embedding_dropout(
            self.token_embedding(ids) * 16.0 + self.position_embedding(positions)
        )
        causal = torch.triu(
            torch.ones(t, t, dtype=torch.bool, device=ids.device), diagonal=1
        )
        padding = None if attention_mask is None else ~attention_mask.bool()
        hidden = self.transformer_decoder(
            target, memory, tgt_mask=causal, tgt_key_padding_mask=padding
        )
        return F.linear(self.final_norm(hidden), self.token_embedding.weight, self.output_bias)

full_model = SiglipModel.from_pretrained(str(MODEL_SOURCE), local_files_only=True)
encoder = full_model.vision_model
del full_model
adapter = SiglipVisualAdapter()
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
assert checkpoint["schema_version"] == "phase3_teacher_c_seed42_checkpoint_v1"
encoder_state = encoder.state_dict()
encoder_state.update(checkpoint["encoder_candidate_state"])
encoder.load_state_dict(encoder_state, strict=True)
adapter.load_state_dict(checkpoint["adapter_state"], strict=True)
del checkpoint

torch.manual_seed(42)
decoder = CaptionDecoder(7456)
encoder, adapter, decoder = encoder.to(DEVICE), adapter.to(DEVICE), decoder.to(DEVICE)
encoder.eval(); adapter.eval(); decoder.train()
for module in (encoder, adapter):
    for parameter in module.parameters(): parameter.requires_grad_(False)

loader = DataLoader(
    TrainPilot(), batch_size=8, shuffle=True,
    generator=torch.Generator().manual_seed(42),
    num_workers=2, pin_memory=True, collate_fn=collate,
)
optimizer = torch.optim.AdamW(
    decoder.parameters(), lr=3e-4, betas=(0.9, 0.999), eps=1e-8, weight_decay=1e-4
)
scaler = torch.amp.GradScaler("cuda", init_scale=1024.0)
ACCUMULATION = 8
TARGET_STEPS = 100
optimizer.zero_grad(set_to_none=True)
started = time.perf_counter()
micro_batches = 0
valid_tokens = 0
last_loss = None
for images, tokens, mask in loader:
    images, tokens, mask = images.to(DEVICE), tokens.to(DEVICE), mask.to(DEVICE)
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        patches = encoder(pixel_values=images).last_hidden_state
        memory = adapter(patches)
    with torch.autocast("cuda", dtype=torch.float16):
        logits = decoder(tokens[:, :-1], memory, mask[:, :-1])
        targets, valid = tokens[:, 1:], mask[:, 1:]
        raw = F.cross_entropy(
            logits.float().reshape(-1, 7456), targets.reshape(-1), reduction="none"
        ).reshape_as(targets)
        loss = (raw * valid).sum() / valid.sum()
    assert logits.shape[-1] == 7456 and torch.isfinite(loss)
    scaler.scale(loss / ACCUMULATION).backward()
    micro_batches += 1
    valid_tokens += int(valid.sum())
    last_loss = float(loss.detach())
    if micro_batches % ACCUMULATION == 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(decoder.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        optimizer.zero_grad(set_to_none=True)
        step = micro_batches // ACCUMULATION
        if step % 10 == 0:
            print(f"optimizer_step={step:03d}/{TARGET_STEPS} | loss={last_loss:.6f}", flush=True)
        if step >= TARGET_STEPS:
            break
elapsed = time.perf_counter() - started
assert micro_batches // ACCUMULATION == TARGET_STEPS
images_processed = micro_batches * 8
rate = images_processed / elapsed
steps_per_epoch = math.ceil(math.ceil(29000 / 8) / ACCUMULATION)
estimated_epoch_hours = (29000 / rate) / 3600

# Pilot weights are deliberately discarded. Full adaptation must restart from
# the locked seed-42 Teacher checkpoint and the same decoder initialization.
RESULT = {
    "schema_version": "f30k_t3_adaptation_pilot_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "passed", "pilot_gate": "full_decoder_adaptation_authorized",
    "teacher": "SigLIP ViT-B/16 + Flickr30k decoder",
    "seed": 42, "source_vocabulary_size": 10080,
    "target_vocabulary_size": 7456, "logit_shape_verified": True,
    "pilot_optimizer_steps": TARGET_STEPS,
    "physical_batch": 8, "gradient_accumulation": ACCUMULATION,
    "effective_batch": 64, "images_processed": images_processed,
    "images_per_second": rate, "optimizer_steps_per_epoch": steps_per_epoch,
    "estimated_train_epoch_hours_excluding_validation": estimated_epoch_hours,
    "last_pilot_ce": last_loss, "pilot_weights_discarded": True,
    "checkpoint_sha256": sha256(CHECKPOINT_PATH),
    "next_phase_authorized": "F30K_T3_Adapt42_Full",
    "test_set_status": "sealed", "test_set_used": False,
}
print("Optimizer steps         :", TARGET_STEPS)
print("Logit vocabulary        : 7,456 VERIFIED")
print("Images/second           :", round(rate, 4))
print("Estimated train epoch   :", round(estimated_epoch_hours, 2), "hours")


[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

optimizer_step=010/100 | loss=7.202159
optimizer_step=020/100 | loss=6.446879
optimizer_step=030/100 | loss=5.009198
optimizer_step=040/100 | loss=4.890343
optimizer_step=050/100 | loss=4.530176
optimizer_step=060/100 | loss=5.066813
optimizer_step=070/100 | loss=4.747947
optimizer_step=080/100 | loss=4.970395
optimizer_step=090/100 | loss=4.135568
optimizer_step=100/100 | loss=4.397691
Optimizer steps         : 100
Logit vocabulary        : 7,456 VERIFIED
Images/second           : 11.5179
Estimated train epoch   : 0.7 hours


## Cell 3 — Result và manifest SHA-256

In [3]:

RESULT_PATH = WORK_ROOT / f"{EXPERIMENT_ID}_RESULT.json"
write_json(RESULT_PATH, RESULT)
artifacts = []
for path in sorted(p for p in WORK_ROOT.rglob("*") if p.is_file() and p.name != "MANIFEST.json"):
    artifacts.append({
        "relative_path": path.relative_to(WORK_ROOT).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256(path),
    })
MANIFEST_PATH = WORK_ROOT / "MANIFEST.json"
write_json(
    MANIFEST_PATH,
    {
        "schema_version": "f30k_teacher_workload_pilot_manifest_v1",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "experiment_id": EXPERIMENT_ID,
        "status": RESULT["status"],
        "artifacts": artifacts,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)
print("=" * 78)
print(EXPERIMENT_ID, "— FINAL")
print("=" * 78)
print("Status                  :", RESULT["status"])
print("Pilot gate              :", RESULT["pilot_gate"])
print("Next                    :", RESULT["next_phase_authorized"])
print("Held-out Test           : SEALED / NOT READ")
print("Output                  :", WORK_ROOT)


F30K_T3_Adapt42_Pilot — FINAL
Status                  : passed
Pilot gate              : full_decoder_adaptation_authorized
Next                    : F30K_T3_Adapt42_Full
Held-out Test           : SEALED / NOT READ
Output                  : /kaggle/working/F30K_T3_Adapt42_Pilot
